# Lord of the Rings NLP Refactor
This notebook demonstrates the refactored LotR NLP pipeline using `lotr_nlp.py`.
It covers text loading, preprocessing, topic modeling, and Word2Vec analysis.

In [ ]:
from pathlib import Path
import lotr_nlp

lotr_nlp.ensure_nltk_resources()

text_path = Path("the_lord_of_the_rings_text/Lord_of_the_Rings_complete.txt")
raw_text = lotr_nlp.load_text(text_path)
print(f"Loaded text length: {len(raw_text)} characters")

## Preprocessing
Tokenize, remove stopwords, and lemmatize the full LotR text.

In [ ]:
extra_stopwords = ["said", "come", "came"]
tokens = lotr_nlp.preprocess_text(raw_text, preserve_case=False, extra_stopwords=extra_stopwords)
print(f"Token count after preprocessing: {len(tokens)}")
print("Sample tokens:", tokens[:30])

## Topic Modeling
Create document chunks, build a term matrix, and fit LDA + NMF models.

In [ ]:
sentences = lotr_nlp.build_sentences(raw_text)
documents = lotr_nlp.chunk_documents(sentences, chunk_size=40)
print(f"Built {len(documents)} document chunks for topic modeling")

matrix, vectorizer = lotr_nlp.build_document_term_matrix(documents)
feature_names = vectorizer.get_feature_names_out()

lda = lotr_nlp.fit_lda(matrix, n_components=10)
print("\nLDA topics:")
lotr_nlp.print_top_words(lda, feature_names, n_top_words=15)

nmf = lotr_nlp.fit_nmf(matrix, n_components=10)
print("\nNMF topics:")
lotr_nlp.print_top_words(nmf, feature_names, n_top_words=15)

## Word2Vec Analysis
Train both CBOW and SkipGram models on sentence-level tokenized text.

In [ ]:
model_cbow = lotr_nlp.train_word2vec(sentences, sg=0)
model_skipgram = lotr_nlp.train_word2vec(sentences, sg=1)

for term in ["ring", "gollum", "frodo"]:
    print(f"\nTop similar words for '{term}' (CBOW):")
    print(model_cbow.wv.most_similar(term, topn=10))
    print(f"\nTop similar words for '{term}' (SkipGram):")
    print(model_skipgram.wv.most_similar(term, topn=10))